# 07 — Logit Lens: Watching Predictions Form Layer by Layer

The **logit lens** is a simple but powerful interpretability technique. The idea: at any intermediate layer of a transformer, we can take the residual stream and apply the model's final layer norm + unembedding matrix to project it into vocabulary space. This gives us a distribution over tokens — what the model *would* predict if it stopped processing at that layer.

By doing this at every layer, we can watch how the model's predictions **evolve** through the network:
- Do early layers already "know" the answer, or do they produce generic tokens?
- At which layer does the correct prediction first appear?
- How does confidence build up over depth?

This reveals that transformers don't compute answers all at once — they **gradually refine** predictions as information flows through successive layers.

In [ ]:
import sys
sys.path.insert(0, "..")
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from utils.model_loading import load_tlens_model, get_tokenizer
from utils.visualization import apply_theme, ACCENT_BLUE, ACCENT_ORANGE, ACCENT_GREEN, ACCENT_RED, ACCENT_PURPLE, ACCENT_TEAL, TEXT_COLOR, DARK_BG, DARK_SURFACE, DARK_GRID, PALETTE, _style_box
apply_theme()

MODEL_SIZE = "0.5b"
model = load_tlens_model(MODEL_SIZE)
tokenizer = get_tokenizer(MODEL_SIZE)

## Implementing the Logit Lens

In [ ]:
@torch.no_grad()
def logit_lens(model, prompt):
    """Apply the logit lens: project each layer's residual stream into vocabulary space."""
    tokens = model.to_str_tokens(prompt)
    logits, cache = model.run_with_cache(prompt)
    n_layers = model.cfg.n_layers
    n_pos = len(tokens)

    all_top_tokens = []
    all_confidences = np.zeros((n_layers, n_pos))

    for layer in range(n_layers):
        residual = cache[f"blocks.{layer}.hook_resid_post"][0]  # [seq, d_model]
        normed = model.ln_final(residual)
        layer_logits = model.unembed(normed)  # [seq, vocab]
        probs = torch.softmax(layer_logits, dim=-1)
        top_probs, top_ids = probs.max(dim=-1)

        layer_tokens = [model.to_string(top_ids[pos].item()) for pos in range(n_pos)]
        all_top_tokens.append(layer_tokens)
        all_confidences[layer] = top_probs.float().cpu().numpy()

    return all_top_tokens, all_confidences, tokens

## Factual Recall: "The capital of France is"

In [ ]:
prompt = "The capital of France is"
top_tokens, confidences, input_tokens = logit_lens(model, prompt)

n_layers, n_pos = confidences.shape

fig, ax = plt.subplots(figsize=(max(n_pos * 1.5, 8), max(n_layers * 0.5, 6)))
im = ax.imshow(confidences, cmap="inferno", aspect="auto", vmin=0, vmax=1)

# Annotate each cell with the predicted token
for layer in range(n_layers):
    for pos in range(n_pos):
        tok = top_tokens[layer][pos].strip()
        if len(tok) > 8:
            tok = tok[:8]
        color = "white" if confidences[layer, pos] > 0.6 else "black"
        ax.text(pos, layer, tok, ha="center", va="center", fontsize=7, color=color)

ax.set_yticks(range(n_layers))
ax.set_yticklabels([f"L{i}" for i in range(n_layers)])
ax.set_xticks(range(n_pos))
ax.set_xticklabels([t.replace(" ", "·") for t in input_tokens], rotation=45, ha="right", fontsize=9)
ax.set_xlabel("Input Token Position")
ax.set_ylabel("Layer")
ax.set_title(f'Logit Lens: "{prompt}"')
plt.colorbar(im, ax=ax, label="Top-1 Probability")
plt.tight_layout()
plt.show()

## Confidence Curve: When Does the Model "Know" the Answer?

In [ ]:
@torch.no_grad()
def confidence_curve(model, prompt, target_token=" Paris"):
    """Track probability of a specific target token across layers at the last position."""
    tokens = model.to_str_tokens(prompt)
    logits, cache = model.run_with_cache(prompt)
    n_layers = model.cfg.n_layers

    # Get target token ID (skip BOS)
    target_id = model.to_tokens(target_token)[0, 1].item()
    target_str = model.to_string(target_id)
    last_pos = len(tokens) - 1

    target_probs = []
    top1_tokens = []

    for layer in range(n_layers):
        residual = cache[f"blocks.{layer}.hook_resid_post"][0]  # [seq, d_model]
        normed = model.ln_final(residual)
        layer_logits = model.unembed(normed)  # [seq, vocab]
        probs = torch.softmax(layer_logits[last_pos], dim=-1)

        target_probs.append(probs[target_id].float().cpu().item())
        top1_tokens.append(probs.argmax().item())

    target_probs = np.array(target_probs)

    # Find first layer where target is top-1
    first_top1 = None
    for i, tid in enumerate(top1_tokens):
        if tid == target_id:
            first_top1 = i
            break

    # Plot
    fig, ax = plt.subplots(figsize=(10, 5))
    layers = range(n_layers)
    ax.plot(layers, target_probs, "o-", color=ACCENT_BLUE, linewidth=2, markersize=5)
    ax.fill_between(layers, target_probs, alpha=0.15, color=ACCENT_BLUE)

    if first_top1 is not None:
        ax.plot(first_top1, target_probs[first_top1], "o", color=ACCENT_RED,
                markersize=12, zorder=5, label=f'"{target_str}" first becomes top-1')
        ax.axvline(first_top1, color=ACCENT_RED, linestyle="--", alpha=0.3)
        ax.legend(fontsize=10)

    ax.set_xlabel("Layer")
    ax.set_ylabel(f'P("{target_str}")')
    ax.set_title(f'Probability of "{target_str}" at last position across layers')
    ax.set_xticks(range(n_layers))
    ax.set_xticklabels([f"L{i}" for i in range(n_layers)])
    ax.set_ylim(-0.02, 1.02)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

confidence_curve(model, "The capital of France is", target_token=" Paris")

## Comparing Different Prompt Types

In [ ]:
prompts = [
    "The Eiffel Tower is located in",
    "The cats are",
    'def hello():\n    print("Hello',
]
prompt_labels = ["Factual", "Syntax", "Code"]

fig, axes = plt.subplots(1, 3, figsize=(20, max(model.cfg.n_layers * 0.5, 6)))

for idx, (prompt, label) in enumerate(zip(prompts, prompt_labels)):
    top_tokens, confidences, input_tokens = logit_lens(model, prompt)
    n_layers, n_pos = confidences.shape
    ax = axes[idx]

    im = ax.imshow(confidences, cmap="inferno", aspect="auto", vmin=0, vmax=1)

    for layer in range(n_layers):
        for pos in range(n_pos):
            tok = top_tokens[layer][pos].strip()
            if len(tok) > 8:
                tok = tok[:8]
            color = "white" if confidences[layer, pos] > 0.6 else "black"
            ax.text(pos, layer, tok, ha="center", va="center", fontsize=6, color=color)

    ax.set_yticks(range(n_layers))
    ax.set_yticklabels([f"L{i}" for i in range(n_layers)], fontsize=7)
    ax.set_xticks(range(n_pos))
    ax.set_xticklabels([t.replace(" ", "·") for t in input_tokens], rotation=45, ha="right", fontsize=7)
    ax.set_title(f"{label}: \"{prompt[:30]}...\"" if len(prompt) > 30 else f'{label}: "{prompt}"', fontsize=9)

    if idx == 0:
        ax.set_ylabel("Layer")

plt.colorbar(im, ax=axes, label="Top-1 Probability", shrink=0.8)
plt.suptitle("Logit Lens Across Prompt Types", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## What the Logit Lens Reveals

**Key observations:**

- **Early layers produce generic tokens.** The first few layers tend to predict high-frequency tokens (articles, punctuation, common words) regardless of context. The residual stream hasn't accumulated enough information yet.

- **Factual knowledge emerges in middle layers.** For prompts like "The capital of France is", the correct answer ("Paris") typically appears partway through the network — not at the very end. The middle layers are where retrieval happens.

- **Final layers refine and sharpen.** The last few layers increase confidence on the already-correct prediction, or make final adjustments. They act more like a refinement pass than a computation pass.

- **The model builds answers gradually.** There is no single "aha" layer. Predictions shift incrementally, which is consistent with the residual stream as a shared communication channel that accumulates information.

**Limitation:** The logit lens assumes that the residual stream is always "ready" to be projected into vocabulary space — that the unembedding matrix is the right decoder at every layer. In practice, intermediate layers may use representations that don't align well with the unembedding. The **Tuned Lens** (Belrose et al., 2023) addresses this by learning a per-layer affine transformation, giving cleaner intermediate predictions.